# Dự báo Doanh thu Sản phẩm
## 2. Kiểm tra Dữ liệu

Trước khi làm sạch và xây dựng mô hình, chúng ta tiến hành kiểm tra kỹ lưỡng tính toàn vẹn, tính hợp lệ logic của các biến số và phạm vi thời gian của tập dữ liệu giao dịch.

Các nội dung kiểm tra:
- Kích thước tập dữ liệu và kiểu dữ liệu của các cột
- Thống kê giá trị khuyết thiếu (null) trên từng cột
- Kiểm tra các hóa đơn bị trùng lặp
- Kiểm tra tính hợp lệ logic của biến số liên tục (tuổi, số lượng, đơn giá)
- Kiểm tra tính liên tục của thời gian ghi nhận hóa đơn

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')

RAW_DATA_PATH = r'../../data/raw-data/customer_shopping_data.csv'
df = pd.read_csv(RAW_DATA_PATH)
print('Kích thước tập dữ liệu:', df.shape)
df.info()

### 2.1 Kiểm tra giá trị khuyết thiếu (Null Values)

In [ ]:
null_counts = df.isnull().sum()
null_pct = (null_counts / len(df) * 100).round(2)
null_df = pd.DataFrame({'Số lượng khuyết': null_counts, 'Tỷ lệ %': null_pct})
print(null_df)

### 2.2 Kiểm tra dữ liệu trùng lặp

In [ ]:
n_dup = df.duplicated().sum()
print(f'Số dòng trùng lặp: {n_dup} ({n_dup / len(df) * 100:.2f}%)')

### 2.3 Kiểm tra miền giá trị và tính hợp lệ logic
Chúng ta kiểm tra xem các trường số liệu có chứa giá trị vô lý không:
- Tuổi khách hàng phải nằm trong phạm vi sinh học hợp lý ($18 \le \text{age} \le 100$).
- Số lượng sản phẩm mua phải lớn hơn 0 (quantity $> 0$).
- Đơn giá sản phẩm phải lớn hơn 0 (price $> 0$).

In [ ]:
invalid_age = df[(df['age'] < 18) | (df['age'] > 100)]
print(f'Số lượng khách hàng có tuổi không hợp lệ (< 18 hoặc > 100): {len(invalid_age)} dòng')

invalid_quantity = df[df['quantity'] <= 0]
print(f'Số lượng hóa đơn có số lượng <= 0: {len(invalid_quantity)} dòng')

invalid_price = df[df['price'] <= 0]
print(f'Số lượng hóa đơn có đơn giá <= 0: {len(invalid_price)} dòng')

### 2.4 Kiểm tra tính liên tục theo thời gian
Chúng ta chuyển đổi trường ngày tháng để xác định phạm vi ghi nhận dữ liệu và tần suất giao dịch hàng tháng.

In [ ]:
df_temp = df.copy()
df_temp['invoice_date'] = pd.to_datetime(df_temp['invoice_date'], format='%d/%m/%Y')
print('Ngày hóa đơn sớm nhất:', df_temp['invoice_date'].min())
print('Ngày hóa đơn muộn nhất:', df_temp['invoice_date'].max())

df_temp['YearMonth'] = df_temp['invoice_date'].dt.to_period('M')
monthly_counts = df_temp.groupby('YearMonth').size()
print('\nSố lượng hóa đơn ghi nhận theo từng tháng:')
print(monthly_counts)

### 2.5 Tổng kết kết quả kiểm tra dữ liệu

| Tiêu chí | Kết quả | Chi tiết phát hiện |
|----------|---------|---------------------|
| Giá trị khuyết thiếu | Đạt | Không phát hiện ô dữ liệu rỗng nào trong bộ dữ liệu. |
| Trùng lặp hóa đơn | Đạt | Không phát hiện dòng dữ liệu nào bị trùng lặp hoàn toàn. |
| Miền tuổi khách hàng | Đạt | Toàn bộ khách hàng nằm trong độ tuổi lao động từ 18 đến 69 tuổi. |
| Số lượng / Đơn giá | Đạt | Không phát hiện giao dịch hoàn hàng hay lỗi nhập đơn giá âm/bằng không. |
| Tính đầy đủ thời gian | Cần xử lý | Tháng 3/2023 chỉ ghi nhận 3 ngày dữ liệu dở dang, dẫn đến tổng số hóa đơn sụt giảm đột biến. |

Chuyển sang **Bước 3: Làm sạch dữ liệu**.